<a href="https://colab.research.google.com/github/your-org/alexpose/blob/main/experiments/multiple-sclerosis/05_representation_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 - Inspect training representations

These pictures use only the teaching fold's training clips. A plot can suggest a pattern, but it cannot measure performance on unseen sources. Camera conditions and pose-detector confidence may explain a cluster, so we compare learned features with visibility features. Validation and test clips are excluded from embeddings, projection fitting, and silhouette scores.

In [ ]:
# --- Setup: install dependencies (Colab installs; local usually already has them) ---
import importlib, importlib.util, subprocess, sys, os

IN_COLAB = 'google.colab' in sys.modules

def _need(mod):
    return importlib.util.find_spec(mod) is None

# Light deps used by every notebook.
_pkgs = []
for mod, pip_name in [('cv2','opencv-python'), ('mediapipe','mediapipe'),
                      ('sklearn','scikit-learn'), ('pandas','pandas'),
                      ('matplotlib','matplotlib'), ('tqdm','tqdm')]:
    if _need(mod):
        _pkgs.append(pip_name)
# torch is guarded so Colab's preinstalled GPU torch is never downgraded.
if _need('torch'):
    _pkgs.append('torch')
if _pkgs:
    print('installing:', _pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs])
else:
    print('all light dependencies already present')

In [ ]:
# --- Make `sjepa` and `ambient` importable, locally and in Colab ---
from pathlib import Path
import sys, subprocess

def _find_exp_dir():
    # Local run: this notebook sits in experiments/multiple-sclerosis.
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / 'sjepa' / '__init__.py').exists():
            return p
    return None

EXP_DIR = _find_exp_dir()
if EXP_DIR is None:
    # Colab: clone the repo, then point at the experiment folder.
    REPO = 'https://github.com/your-org/alexpose.git'  # <-- edit to your fork
    if not Path('alexpose').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO])
    EXP_DIR = Path('alexpose') / 'experiments' / 'multiple-sclerosis'

REPO_ROOT = EXP_DIR.parents[1]
for p in (str(EXP_DIR), str(REPO_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
print('experiment dir:', EXP_DIR)
print('repo root     :', REPO_ROOT)

In [ ]:
# --- Paths and profile (reads the root .env if python-dotenv is present) ---
import os
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / '.env')
except Exception:
    pass

VIDEO_DIR = EXP_DIR / 'video-data-full'
ARTIFACT_DIR = EXP_DIR / 'artifacts'
KEYPOINTS_DIR = ARTIFACT_DIR / 'keypoints-full'
IMAGES_DIR = EXP_DIR / 'images'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Pick the model size profile. 'laptop' is the fast default; set SJEPA_PROFILE=gpu
# in your .env for a larger model, or SJEPA_SMOKE=1 for a near-instant test run.
os.environ.setdefault('SJEPA_PROFILE', 'laptop')
print('SJEPA_PROFILE =', os.environ['SJEPA_PROFILE'],
      '| SJEPA_SMOKE =', os.environ.get('SJEPA_SMOKE', '0'))

## Keep each source video together

We use the same frozen five-fold registry in notebooks 02–06. A source video may have several clips; each clip may yield overlapping windows. All of those relatives stay together. Each round uses about 60% of sources for training, 20% for validation, and 20% for testing. Only notebook 06 evaluates test clips.

The splitter runs on one row per source, with condition labels used to balance source counts. It never splits windows. The loader checks the full cache, reviewed exclusions, and registry checksum. A changed cache requires a new registry and new checkpoints. See [the full method](docs/11-full-data-splits.md).

In [ ]:
from IPython.display import display
import pandas as pd
from sjepa.splits import load_full_registry, partition_records, split_summary
records, registry = load_full_registry(EXP_DIR)
FOLD = 0  # teaching example; notebook 06 independently trains all five folds
train_recs, val_recs, test_recs = partition_records(records, registry, FOLD)
display(pd.DataFrame(split_summary(records, registry)))
print('usable clips:', len(records), '| excluded raw clips:', len(registry['inventory']['exclusions']))
print('registry:', registry['registry_sha256'])

In [ ]:
import numpy as np
from sjepa.config import get_config
from sjepa.models import build_model, pick_device
from sjepa.splits import fold_run_dir, load_partition_checkpoint
from sjepa.full_experiment import embed_records, nuisance_features
cfg = get_config(); device = pick_device()
RUN_DIR = fold_run_dir(EXP_DIR, registry, cfg, FOLD)
representations = {}
for stage in ['ssl', 'continued']:
    model = build_model(cfg, device=device, repaired=True)
    load_partition_checkpoint(RUN_DIR / f'{stage}.pt', model, cfg, registry, FOLD, stage, device)
    representations[stage] = embed_records(model, train_recs, cfg, device)
representations['visibility'] = nuisance_features(train_recs)
y = [r.label for r in train_recs]
np.savez(RUN_DIR / 'training_embeddings.npz', **representations, labels=np.array(y),
         clips=np.array([r.clip_name for r in train_recs]),
         sources=np.array([r.source_id for r in train_recs]),
         registry_sha256=registry['registry_sha256'], partition='train')
print('training clips plotted:', len(y))

## Project training points to two dimensions

t-SNE and UMAP can distort distances and create apparent clusters. We standardize each feature using these training clips, then fit the projection on the same training clips. Colors show dataset labels. Multiple points from one source are related observations.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sjepa.viz import scatter_2d
import matplotlib.pyplot as plt
scaled = {name: StandardScaler().fit_transform(E) for name, E in representations.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, E) in zip(axes, scaled.items()):
    xy = TSNE(n_components=2, perplexity=min(15, len(E)-1), random_state=42, init='pca').fit_transform(E)
    scatter_2d(xy, y, ax, f'training only: {name}')
plt.tight_layout(); plt.savefig(RUN_DIR / 'training_tsne.png', dpi=130); plt.show()

In [ ]:
try:
    import umap
except ImportError:
    print('Install umap-learn to enable the optional UMAP view.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (name, E) in zip(axes, scaled.items()):
        xy = umap.UMAP(n_neighbors=min(15, len(E)-1), min_dist=0.3, random_state=42).fit_transform(E)
        scatter_2d(xy, y, ax, f'training UMAP: {name}')
    plt.tight_layout(); plt.show()

In [ ]:
from sjepa.eval import silhouette
for name, E in scaled.items():
    print(f'{name}: training-only silhouette = {silhouette(E, y):.3f}')

A training silhouette is descriptive. It is not an out-of-fold score or proof that the encoder learned a clinical feature. Keep the evaluation choices fixed before notebook 06. Changing them after exploring this collection makes the study exploratory.